In [45]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from COMPAS.compas_python_utils.detailed_evolution_plotter import plot_detailed_evolution as det_evo
import tempfile

import h5py as h5 
from astropy import units as u
from astropy import constants as c

import os
import scipy

pd.options.display.max_columns = None

In [46]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import matplotlib
from matplotlib.ticker import LogLocator, AutoMinorLocator, MultipleLocator
# rcParams['font.family'] = 'serif'
# rcParams["mathtext.fontset"] = 'stix'
# rcParams["font.size"] = 18
fontparams = {
    "font.family": "serif",
    "mathtext.fontset" : "stix",
    "grid.color": "gray",
    "grid.linestyle": ":",
    "axes.titlesize": "18",
    "axes.labelsize": "12",
    "xtick.labelsize": "10",
    "ytick.labelsize": "10",
    "xtick.labelbottom": "True",
    "legend.framealpha": "1",
}
rcParams.update(fontparams) 

%config InlineBackend.figure_format='retina' # very useful command for high-res images


In [47]:
from cycler import cycler

colorPalette = {'red': "#E64D4E",
                'orange': "#EE9063",
                'yellow': "#FFDD7B",
                'green': "#77AC54",
                'blue': "#0B92B1",
                'violet': "#665191",
                'gray': "#B4B4B4"
}

custom_cycler = (cycler(color=[colorPalette['red'], colorPalette['blue'], colorPalette['green']]))
color_list = [colorPalette['red'], colorPalette['blue'], colorPalette['green']]

st_labels = ['MS_LT_0.7', 'MS_GT_0.7', 'HG', 'FGB', 'CHeB', 'EAGB', 'TPAGB', 'HeMS', 'HeHG', 'HeGB', 'HeWD', 'COWD', 'ONeWD', 'NS', 'BH', 'MR','CHE',  '--', '--', 'None']


In [49]:
columns_to_keep = ['Time', 'Mass(1)', 'Mass(2)', 'Radius(1)', 'Radius(2)', 'SemiMajorAxis', 'Eccentricity', 'Stellar_Type(1)', 'Stellar_Type(2)', 'Period(1)', 'Period(2)', 'Period_Orb', 'dmMT(1)', 'dmMT(2)', 'Omega(1)', 'Omega(2)', 'Omega_Orb', 'Record_Type']
columns_to_keep_sanity = ['Time', 'M1', 'M2', 'R1', 'R2', 'SemiMajorAxis_Rsun', 'ecc', 'StellarType_1', 'StellarType_2', 'Period1', 'Period2', 'Period_orb', 'J1_after', 'J2_after', 'Jorb_after', 'dM1_MT', 'dM2_MT', 'ImK22_1', 'ImK22_2', 'ImK12_1_dyn', 'ImK12_2_dyn', 'ImK12_1_eq', 'ImK12_2_eq']


1. INITIAL_STATE
    Record describes the initial state of the binary

2. POST_STELLAR_TIMESTEP
    Record was logged immediately following stellar timestep (i.e. the evolution of the constituent stars for a single timestep)

3.  POST_BINARY_TIMESTEP
    Record was logged immediately following binary timestep (i.e. the evolution of the binary system for a single timestep)

4.  TIMESTEP_COMPLETED
    Record was logged immediately following the completion of the timestep (after all changes to the binary and components)

5.  FINAL_STATE
    Record describes the final state of the binary

6.  STELLAR_TYPE_CHANGE_DURING_CEE
    Record was logged immediately following a stellar type change during a common envelope event

7.  STELLAR_TYPE_CHANGE_DURING_MT
    Record was logged immediately following a stellar type change during a mass transfer event

8.  STELLAR_TYPE_CHANGE_DURING_MASS_RESOLUTION
    Record was logged immediately following a stellar type change during mass resolution

9.  STELLAR_TYPE_CHANGE_DURING_CHE_EQUILIBRATION
    Record was logged immediately following a stellar type change during mass equilibration for CHE

10.  POST_MT
    Record was logged immediately following a mass transfer event

11.  POST_WINDS
    Record was logged immediately following winds mass loss

12.  POST_CEE
    Record was logged immediately following a common envelope event

13.  POST_SN
    Record was logged immediately following a supernova event

14.  POST_MASS_RESOLUTION
    Record was logged immediately following mass resolution (i.e. after winds mass loss & mass transfer complete)

15.  POST_MASS_RESOLUTION_MERGER
    Record was logged immediately following a merger after mass resolution

16.  PRE_STELLAR_TIMESTEP
    Record was logged immediately prior to stellar timestep (i.e. the evolution of the constituent stars for a single timestep)


# SMT BBH that stays narrow / spinning with Tides

In [50]:
# notides_DataPath = 'sim_data/smt_notides'
tides_DataPath = 'sim_data/smt_realistic'
perfect_DataPath = 'sim_data/smt_perfect'

# # --------------

# df_detailed_notides = pd.read_csv(notides_DataPath+'/Detailed_Output/BSE_Detailed_Output_0.csv', skiprows=2, header=0)
# df_detailed_notides = df_detailed_notides.rename(columns=lambda x: x.strip())

df_detailed_tides = pd.read_csv(tides_DataPath+'/Detailed_Output/BSE_Detailed_Output_0.csv', skiprows=2, header=0)
df_detailed_tides = df_detailed_tides.rename(columns=lambda x: x.strip())

df_detailed_perfect = pd.read_csv(perfect_DataPath+'/Detailed_Output/BSE_Detailed_Output_0.csv', skiprows=2, header=0)
df_detailed_perfect = df_detailed_perfect.rename(columns=lambda x: x.strip())



# df_dco_notides = pd.read_csv(notides_DataPath+'/BSE_Double_Compact_Objects.csv', skiprows=2, header=0)
# df_dco_notides = df_dco_notides.rename(columns=lambda x: x.strip())

df_dco_tides = pd.read_csv(tides_DataPath+'/BSE_Double_Compact_Objects.csv', skiprows=2, header=0)
df_dco_tides = df_dco_tides.rename(columns=lambda x: x.strip())

df_dco_perfect = pd.read_csv(perfect_DataPath+'/BSE_Double_Compact_Objects.csv', skiprows=2, header=0)
df_dco_perfect = df_dco_perfect.rename(columns=lambda x: x.strip())


### Realistic Tides

In [51]:
RawData = df_detailed_tides
tf = tempfile.TemporaryFile()
Data = h5.File(tf, 'w')
maskRecordType4 = RawData['Record_Type'] == 4     # Filter first for only end-of-timestep events
for key in RawData.keys():
    Data.create_dataset(key, data=RawData[key][maskRecordType4])

### Collect the important events in the detailed evolution
events = det_evo.allEvents(Data).allEvents                 # Calculate the events here, for use in plot sizing parameters
det_evo.printEvolutionaryHistory(events=events)
# det_evo.makeDetailedPlots(Data, events, outdir=None, show=True)
print("Merges Hubble Time?:", df_dco_tides['Merges_Hubble_Time'].values.astype(bool))
det_evo.plotVanDenHeuvel(events, outdir=None, use_latex=True)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/veomekapil/Library/CloudStorage/OneDrive-JohnsHopkins/Research/tides_paper_2/COMPAS/compas_python_utils/detailed_evolution_plotter/van_den_heuvel_figures/2.png'

In [52]:
df_detailed_tides['Period(1)'] = (2 * np.pi / (df_detailed_tides['Omega(1)'].values / u.s)).to(u.day)
df_detailed_tides['Period(2)'] = (2 * np.pi / (df_detailed_tides['Omega(2)'].values / u.s)).to(u.day)

omega_orb = np.sqrt(c.G * (df_detailed_tides['Mass(1)'].values * u.M_sun + df_detailed_tides['Mass(2)'].values * u.M_sun) / (df_detailed_tides['SemiMajorAxis'].values*u.AU)**3).to(1/u.s)
df_detailed_tides['Omega_Orb'] = omega_orb.value
df_detailed_tides['Period_Orb'] = (2 * np.pi / omega_orb).to(u.day)

df_detailed_tides_short = df_detailed_tides[columns_to_keep]

/opt/anaconda3/envs/tides_compas/lib/python3.11/site-packages/astropy/units/quantity.py:671: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/opt/anaconda3/envs/tides_compas/lib/python3.11/site-packages/astropy/units/quantity.py:671: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)


In [53]:
# st_change_idx = np.where((df_detailed_tides['Stellar_Type(1)'] != df_detailed_tides['Stellar_Type(1)'].shift()) | (df_detailed_tides['Stellar_Type(2)'] != df_detailed_tides['Stellar_Type(2)'].shift()))[0]
# st_changes_with_buffer = np.concatenate([st_change_idx, st_change_idx+1])
# st_changes_with_buffer.sort()
# df_detailed_tides_short.iloc[st_changes_with_buffer] # the +1 is to get the row after the change, which is after tidal effects

In [54]:
df_dco_tides

,SEED,SemiMajorAxis@DCO,Eccentricity@DCO,Mass(1),Stellar_Type(1),Mass(2),Stellar_Type(2),Coalescence_Time,Time,Merges_Hubble_Time,Recycled_NS(1),Recycled_NS(2),Record_Type
0,1135,0.049567,0.0,11.87286,14,16.53033,14,348.143416,9.285365,1,0,0,1


In [55]:
df_sanity_tides = pd.read_csv(tides_DataPath+'/sanity_checks.csv', header=0)
df_sanity_tides = df_sanity_tides.rename(columns=lambda x: x.strip())
df_sanity_tides.head()

,SEED,Time,Dt,StellarType_1,StellarType_2,J_before,J_after,Jorb_before,Jorb_after,J1_before,J1_after,J2_before,J2_after,Omega_before,Omega,Omega1_before,Omega1,Omega2_before,Omega2,ecc_before,ecc,M1,R1,I1_before,I1_after,M2,R2,I2_before,I2_after,SemiMajorAxis_before,SemiMajorAxis_after,ImK10_1,ImK10_2,ImK12_1,ImK12_2,ImK22_1,ImK22_2,ImK32_1,ImK32_2,ImK10_1_dyn,ImK10_2_dyn,ImK12_1_dyn,ImK12_2_dyn,ImK22_1_dyn,ImK22_2_dyn,ImK32_1_dyn,ImK32_2_dyn,ImK10_1_eq,ImK10_2_eq,ImK12_1_eq,ImK12_2_eq,ImK22_1_eq,ImK22_2_eq,ImK32_1_eq,ImK32_2_eq,M1_core,R1_core,M2_core,R2_core,M1_conv_env,M1_conv_env_max,M2_conv_env,M2_conv_env_max,R1_conv_env_extent,R2_conv_env_extent,tau_conv1,tau_conv2,Lum_1,Lum_2,Temp_1,Temp_2,dadt1,dadt2,dOmegadt1,dOmegadt2,dedt1,dedt2,DaDt_tidal,DeDt_tidal,DOmegaDt_tidal,suggested_dt,dM1_winds,dM2_winds,dM1_MT,dM2_MT,ImK10_Zahn_Equilibrium1,ImK10_Zahn_Equilibrium2,ImK12_Zahn_Equilibrium1,ImK12_Zahn_Equilibrium2,ImK22_Zahn_Equilibrium1,ImK22_Zahn_Equilibrium2,ImK32_Zahn_Equilibrium1,ImK32_Zahn_Equilibrium2,ImK10_Zahn_Dynamical1,ImK10_Zahn_Dynamical2,ImK12_Zahn_Dynamical1,ImK12_Zahn_Dynamical2,ImK22_Zahn_Dynamical1,ImK22_Zahn_Dynamical2,ImK32_Zahn_Dynamical1,ImK32_Zahn_Dynamical2,DSemiMajorAxis1Dt_tidal_Zahn,DSemiMajorAxis2Dt_tidal_Zahn,DEccentricity1Dt_tidal_Zahn,DEccentricity2Dt_tidal_Zahn,DOmega1Dt_tidal_Zahn,DOmega2Dt_tidal_Zahn
0,1135,0.000000,0.000056,1,1,374.729,374.729,374.729,374.729,0.000000,0.000037,0.000000,0.000007,261.586,261.586,0.000000,0.012636,0.000000,0.006351,0.350775,0.350774,36.6118,6.10301,0.002949,0.002949,23.6347,4.73158,0.001144,0.001144,0.326353,0.326353,4.833800e-08,1.402600e-08,4.833800e-08,1.402600e-08,3.069270e-07,8.905960e-08,9.049240e-07,2.625780e-07,4.833800e-08,1.402600e-08,4.833800e-08,1.402600e-08,3.069270e-07,8.905960e-08,9.049240e-07,2.625780e-07,0,0,0,0,0,0,0,0,20.7124,3.20676,11.4761,2.22310,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,175845.0,62806.3,8.28917,7.27776,-1.784130e-09,-3.479590e-10,0.000254,0.000127,-2.395400e-09,-4.671740e-10,1.82920,1.46437,0.010313,0.010313,-0.000005,-8.969480e-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.880360e-09,9.118570e-10,4.880360e-09,9.118570e-10,3.098840e-08,5.789930e-09,9.136410e-08,1.707070e-08,-1.801320e-10,-2.262140e-11,-2.418480e-10,-3.037180e-11,0.000026,0.000008
1,1135,0.000056,0.014071,1,1,374.722,374.722,374.722,374.714,0.000037,0.006898,0.000007,0.001345,261.574,261.600,0.012634,2.339030,0.006351,1.175770,0.350774,0.350745,36.6106,6.10292,0.002949,0.002949,23.6344,4.73157,0.001144,0.001144,0.326361,0.326339,4.832420e-08,1.402320e-08,4.831170e-08,1.402140e-08,3.068000e-07,8.903580e-08,9.045880e-07,2.625130e-07,4.832420e-08,1.402320e-08,4.831170e-08,1.402140e-08,3.068000e-07,8.903580e-08,9.045880e-07,2.625130e-07,0,0,0,0,0,0,0,0,20.7114,3.20665,11.4759,2.22307,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,175834.0,62805.1,8.28909,7.27773,-1.783130e-09,-3.478110e-10,0.000254,0.000127,-2.394010e-09,-4.669660e-10,1.83027,1.46522,0.010318,0.010318,-0.001189,-2.253050e-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.879240e-09,9.117210e-10,4.877990e-09,9.116030e-10,3.097730e-08,5.788690e-09,9.133520e-08,1.706740e-08,-1.800410e-10,-2.261300e-11,-2.417210e-10,-3.035990e-11,0.000026,0.000008
2,1135,0.014127,0.014064,1,1,374.715,374.715,374.707,374.699,0.006896,0.013769,0.001345,0.002704,261.588,261.614,2.334010,4.659960,1.174110,2.359810,0.350745,0.350715,36.6094,6.10891,0.002955,0.002955,23.6342,4.73478,0.001146,0.001146,0.326346,0.326324,4.743010e-08,1.384730e-08,4.520650e-08,1.351830e-08,2.940500e-07,8.687640e-08,8.739120e-07,2.571690e-07,4.743010e-08,1.384730e-08,4.520650e-08,1.351830e-08,2.940500e-07,8.687640e-08,8.739120e-07,2.571690e-07,0,0,0,0,0,0,0,0,20.6897,3.20120,11.4679,2.22053,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,176058.0,62860.5,8.28766,7.27687,-1.730620e-09,-3.418400e-10,0.000246,0.000125,-2.325550e-09,-4.591760e-10,1.88572,1.50822,0.010655,0.010655,-0.001192,-2.256240e-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.898900e-09,9.143190e-10,4.669230e-09,8.925960e-10,3

In [56]:
df_sanity_tides['SemiMajorAxis_Rsun'] = (df_sanity_tides['SemiMajorAxis_after'].values * u.AU).to(u.R_sun)
df_sanity_tides['Period1'] = (2 * np.pi / (df_sanity_tides['Omega1'].values / u.yr)).to(u.day)
df_sanity_tides['Period2'] = (2 * np.pi / (df_sanity_tides['Omega2'].values / u.yr)).to(u.day)
df_sanity_tides['Period_orb'] = (2 * np.pi / (df_sanity_tides['Omega'].values / u.yr)).to(u.day)


df_sanity_tides_short = df_sanity_tides[columns_to_keep_sanity]

In [57]:
st_change_idx = np.where((df_sanity_tides_short['StellarType_1'] != df_sanity_tides_short['StellarType_1'].shift()) | (df_sanity_tides_short['StellarType_2'] != df_sanity_tides_short['StellarType_2'].shift()))[0]
st_changes_with_buffer = np.concatenate([st_change_idx, st_change_idx-1])
st_changes_with_buffer.sort()
df_sanity_tides_short.iloc[st_changes_with_buffer][1:]

,Time,M1,M2,R1,R2,SemiMajorAxis_Rsun,ecc,StellarType_1,StellarType_2,Period1,Period2,Period_orb,J1_after,J2_after,Jorb_after,dM1_MT,dM2_MT,ImK22_1,ImK22_2,ImK12_1_dyn,ImK12_2_dyn,ImK12_1_eq,ImK12_2_eq
0,0.00000,36.6118,23.6347,6.10301,4.73158,70.176389,3.507740e-01,1,1,181622.975652,3.613465e+05,8.773151,0.000037,0.000007,374.7290,0.000000,0.0000,3.069270e-07,8.905960e-08,4.833800e-08,1.402600e-08,0,0
1379,5.65109,35.4554,23.4764,15.22220,7.15803,71.475183,3.489320e-01,1,1,132.877855,5.164175e+01,9.117842,0.306864,0.115606,368.0890,0.000000,0.0000,2.014320e-13,3.795440e-11,2.586730e-14,3.140710e-12,0,0
1380,5.65109,35.4552,23.4764,14.88250,7.15803,71.475398,3.489320e-01,2,1,84.825057,5.164186e+01,9.117914,0.306848,0.115606,368.0880,0.000000,0.0000,2.012570e-13,3.795350e-11,2.251040e-14,3.140620e-12,0,0
1563,5.65216,35.4549,23.4764,19.31470,7.15903,71.475828,3.489320e-01,2,1,142.353234,5.165663e+01,9.118023,0.306816,0.115605,368.0870,0.000000,0.0000,6.289790e-14,3.788080e-11,8.199570e-15,3.135370e-12,0,0
1564,5.65216,11.9395,46.9918,1.04452,8.18978,78.546301,0.000000e+00,7,1,142.478732,3.960346e-01,10.503945,0.000454,39.499900,277.5390,-23.515400,23.5154,0.000000e+00,-1.139790e-04,0.000000e+00,-1.200310e-04,0,0
1752,6.22752,11.8757,46.7997,1.04110,8.85768,80.808224,0.000000e+00,7,1,145.858233,5.040940e-01,10.984747,0.000438,36.152100,279.4670,0.000000,0.0000,0.000000e+00,-1.555830e-05,0.000000e+00,-1.657620e-05,0,0
1753,6.22752,11.8755,46.7989,1.04110,8.85760,80.817900,0.000000e+00,8,1,43.339964,5.042779e-01,10.986851,0.000438,36.137800,279.4760,0.000000,0.0000,0.000000e+00,-1.555470e-05,0.000000e+00,-1.657230e-05,0,0
1828,6.24693,11.8725,46.7924,1.12914,8.88425,80.904558,0.000000e+00,8,1,44.297748,5.090339e-01,11.005397,0.000435,36.010900,279.5390,0.000000,0.0000,0.000000e+00,-1.442900e-05,0.000000e+00,-1.538100e-05,0,0
1829,6.24713,11.8725,46.7923,0.00005,8.88452,80.905419,0.000000e+00,14,1,0.000001,5.090836e-01,11.005608,0.000435,36.009500,279.5400,0.000000,0.0000,0.000000e+00,-1.441810e-05,0.000000e+00,-1.536940e-05,0,0
3036,8.80735,11.8725,45.1744,0.00005,17.94910,88.416277,0.000000e+00,14,1,0.000001,2.635707e+00,12.750197,0.000435,27.405900,286.0960,0.000000,0.0000,0.000000e+00,-6.167500e-12,0.000000e+00,-8.549720e-12,0,0


### Perfect Tides

In [58]:
RawData = df_detailed_perfect
tf = tempfile.TemporaryFile()
Data = h5.File(tf, 'w')
maskRecordType4 = RawData['Record_Type'] == 4     # Filter first for only end-of-timestep events
for key in RawData.keys():
    Data.create_dataset(key, data=RawData[key][maskRecordType4])

### Collect the important events in the detailed evolution
events = det_evo.allEvents(Data).allEvents                 # Calculate the events here, for use in plot sizing parameters
det_evo.printEvolutionaryHistory(events=events)
# det_evo.makeDetailedPlots(Data, events, outdir=None, show=True)
# print("Merges Hubble Time?:", df_dco_perfect['Merges_Hubble_Time'].values.astype(bool))
det_evo.plotVanDenHeuvel(events, outdir=None, use_latex=True)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/veomekapil/Library/CloudStorage/OneDrive-JohnsHopkins/Research/tides_paper_2/COMPAS/compas_python_utils/detailed_evolution_plotter/van_den_heuvel_figures/2.png'

In [59]:
df_detailed_perfect['Period(1)'] = (2 * np.pi / (df_detailed_perfect['Omega(1)'].values / u.s)).to(u.day)
df_detailed_perfect['Period(2)'] = (2 * np.pi / (df_detailed_perfect['Omega(2)'].values / u.s)).to(u.day)

omega_orb = np.sqrt(c.G * (df_detailed_perfect['Mass(1)'].values * u.M_sun + df_detailed_perfect['Mass(2)'].values * u.M_sun) / (df_detailed_perfect['SemiMajorAxis'].values*u.AU)**3).to(1/u.s)
df_detailed_perfect['Omega_Orb'] = omega_orb.value
df_detailed_perfect['Period_Orb'] = (2 * np.pi / omega_orb).to(u.day)

df_detailed_perfect_short = df_detailed_perfect[columns_to_keep]

/opt/anaconda3/envs/tides_compas/lib/python3.11/site-packages/astropy/units/quantity.py:671: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/opt/anaconda3/envs/tides_compas/lib/python3.11/site-packages/astropy/units/quantity.py:671: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)


In [60]:
# st_change_idx = np.where((df_detailed_perfect_short['Stellar_Type(1)'] != df_detailed_perfect_short['Stellar_Type(1)'].shift()) | (df_detailed_perfect_short['Stellar_Type(2)'] != df_detailed_perfect_short['Stellar_Type(2)'].shift()))[0]
# st_changes_with_buffer = np.concatenate([st_change_idx, st_change_idx+1])
# st_changes_with_buffer.sort()
# df_detailed_perfect_short.iloc[st_changes_with_buffer]


In [61]:
df_dco_perfect

,SEED,SemiMajorAxis@DCO,Eccentricity@DCO,Mass(1),Stellar_Type(1),Mass(2),Stellar_Type(2),Coalescence_Time,Time,Merges_Hubble_Time,Recycled_NS(1),Recycled_NS(2),Record_Type
0,1135,0.06684,0.0,11.908244,14,16.557366,14,1143.380613,9.286152,1,0,0,1


In [62]:
df_sanity_perfect = pd.read_csv(perfect_DataPath+'/sanity_checks.csv', header=0)
df_sanity_perfect = df_sanity_perfect.rename(columns=lambda x: x.strip())

In [63]:
df_sanity_perfect['SemiMajorAxis_Rsun'] = (df_sanity_perfect['SemiMajorAxis_after'].values * u.AU).to(u.R_sun)
df_sanity_perfect['Period1'] = (2 * np.pi / (df_sanity_perfect['Omega1'].values / u.yr)).to(u.day)
df_sanity_perfect['Period2'] = (2 * np.pi / (df_sanity_perfect['Omega2'].values / u.yr)).to(u.day)
df_sanity_perfect['Period_orb'] = (2 * np.pi / (df_sanity_perfect['Omega'].values / u.yr)).to(u.day)


df_sanity_perfect_short = df_sanity_perfect[columns_to_keep_sanity]

In [64]:
st_change_idx = np.where((df_sanity_perfect_short['StellarType_1'] != df_sanity_perfect_short['StellarType_1'].shift()) | (df_sanity_perfect_short['StellarType_2'] != df_sanity_perfect_short['StellarType_2'].shift()))[0]
st_changes_with_buffer = np.concatenate([st_change_idx, st_change_idx-1])
st_changes_with_buffer.sort()
df_sanity_perfect_short.iloc[st_changes_with_buffer][1:]


,Time,M1,M2,R1,R2,SemiMajorAxis_Rsun,ecc,StellarType_1,StellarType_2,Period1,Period2,Period_orb,J1_after,J2_after,Jorb_after,dM1_MT,dM2_MT,ImK22_1,ImK22_2,ImK12_1_dyn,ImK12_2_dyn,ImK12_1_eq,ImK12_2_eq
0,0.00000,36.6118,23.6347,6.10301,4.73158,61.109558,0,1,1,7.129093,7.129093,7.129093,9.493760e-01,3.683760e-01,373.4110,0.000000,0.0000,3.069270e-07,8.905960e-08,4.833800e-08,1.402600e-08,0,0
1379,5.65109,35.4554,23.4764,15.22220,7.15803,60.718200,0,1,1,7.139006,7.139006,7.139006,5.711640e+00,8.362650e-01,362.0150,0.000000,0.0000,-2.974500e-20,3.748170e-23,-7.443770e-14,-1.927120e-11,0,0
1380,5.65109,35.4552,23.4764,14.88250,7.15803,61.439848,0,2,1,7.266679,7.266679,7.266679,3.581880e+00,8.215720e-01,364.1590,0.000000,0.0000,-1.149960e-13,-1.320510e-24,-6.213930e-13,-1.926820e-11,0,0
1803,5.65315,35.4545,23.4763,24.56320,7.15996,59.195127,0,2,1,6.872149,6.872149,6.872149,1.025100e+01,8.692040e-01,357.4380,0.000000,0.0000,4.777710e-22,2.368200e-27,-7.605030e-15,-2.123980e-11,0,0
1804,5.65315,11.9753,46.9555,1.04644,8.18793,129.828244,0,7,1,22.321215,22.321215,22.321215,2.915830e-03,6.999700e-01,357.6140,-23.479200,23.4792,0.000000e+00,-1.173910e-04,0.000000e+00,-1.214520e-04,0,0
1992,6.22742,11.9112,46.7641,1.04300,8.85423,130.314002,0,7,1,22.495378,22.495378,22.495378,2.858880e-03,8.088790e-01,355.6840,0.000000,0.0000,0.000000e+00,3.774900e-20,0.000000e+00,-1.104970e-10,0,0
1993,6.22742,11.9109,46.7634,1.04300,8.85416,130.317658,0,8,1,22.496480,22.496480,22.496480,8.479260e-04,8.088120e-01,355.6790,0.000000,0.0000,0.000000e+00,9.970450e-22,0.000000e+00,-1.105440e-10,0,0
2067,6.24662,11.9080,46.7569,1.13006,8.88050,130.335505,0,8,1,22.502877,22.502877,22.502877,8.604890e-04,8.132870e-01,355.5930,0.000000,0.0000,0.000000e+00,4.776770e-21,0.000000e+00,-1.052800e-10,0,0
2068,6.24682,11.9080,46.7568,0.00005,8.88077,130.336365,0,14,1,22.503319,22.503319,22.503319,2.678070e-11,8.133260e-01,355.5940,0.000000,0.0000,0.000000e+00,4.776220e-21,0.000000e+00,-1.052280e-10,0,0
3274,8.80822,11.9080,45.1419,0.00005,17.94040,132.309500,0,14,1,23.339643,23.339643,23.339643,2.582090e-11,3.089680e+00,350.7630,0.000000,0.0000,0.000000e+00,-1.472280e-22,0.000000e+00,-5.379110e-15,0,0


# CE binary not able to spin up

In [71]:
notides_DataPath = 'sim_data/ce_notides'
tides_DataPath = 'sim_data/ce_realistic'
perfect_DataPath = 'sim_data/ce_perfect'

# # --------------

df_detailed_notides = pd.read_csv(notides_DataPath+'/Detailed_Output/BSE_Detailed_Output_0.csv', skiprows=2, header=0)
df_detailed_notides = df_detailed_notides.rename(columns=lambda x: x.strip())

df_detailed_tides = pd.read_csv(tides_DataPath+'/Detailed_Output/BSE_Detailed_Output_0.csv', skiprows=2, header=0, low_memory=False)
df_detailed_tides = df_detailed_tides.rename(columns=lambda x: x.strip())

df_detailed_perfect = pd.read_csv(perfect_DataPath+'/Detailed_Output/BSE_Detailed_Output_0.csv', skiprows=2, header=0)
df_detailed_perfect = df_detailed_perfect.rename(columns=lambda x: x.strip())



df_dco_notides = pd.read_csv(notides_DataPath+'/BSE_Double_Compact_Objects.csv', skiprows=2, header=0)
df_dco_notides = df_dco_notides.rename(columns=lambda x: x.strip())

df_dco_tides = pd.read_csv(tides_DataPath+'/BSE_Double_Compact_Objects.csv', skiprows=2, header=0)
df_dco_tides = df_dco_tides.rename(columns=lambda x: x.strip())

df_dco_perfect = pd.read_csv(perfect_DataPath+'/BSE_Double_Compact_Objects.csv', skiprows=2, header=0)
df_dco_perfect = df_dco_perfect.rename(columns=lambda x: x.strip())


### Realistic Tides

In [72]:
RawData = df_detailed_tides
tf = tempfile.TemporaryFile()
Data = h5.File(tf, 'w')
maskRecordType4 = RawData['Record_Type'] == 4     # Filter first for only end-of-timestep events
for key in RawData.keys():
    Data.create_dataset(key, data=RawData[key][maskRecordType4])

### Collect the important events in the detailed evolution
events = det_evo.allEvents(Data).allEvents                 # Calculate the events here, for use in plot sizing parameters
det_evo.printEvolutionaryHistory(events=events)
# det_evo.makeDetailedPlots(Data, events, outdir=None, show=True)
print("Merges Hubble Time?:", df_dco_tides['Merges_Hubble_Time'].values.astype(bool))
det_evo.plotVanDenHeuvel(events, outdir=None, use_latex=True)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/veomekapil/Library/CloudStorage/OneDrive-JohnsHopkins/Research/tides_paper_2/COMPAS/compas_python_utils/detailed_evolution_plotter/van_den_heuvel_figures/2.png'

In [73]:
df_sanity_tides = pd.read_csv(tides_DataPath+'/sanity_checks.csv', header=0)
df_sanity_tides = df_sanity_tides.rename(columns=lambda x: x.strip())

df_sanity_tides['SemiMajorAxis_Rsun'] = (df_sanity_tides['SemiMajorAxis_after'].values * u.AU).to(u.R_sun)
df_sanity_tides['Period1'] = (2 * np.pi / (df_sanity_tides['Omega1'].values / u.yr)).to(u.day)
df_sanity_tides['Period2'] = (2 * np.pi / (df_sanity_tides['Omega2'].values / u.yr)).to(u.day)
df_sanity_tides['Period_orb'] = (2 * np.pi / (df_sanity_tides['Omega'].values / u.yr)).to(u.day)

df_sanity_tides_short = df_sanity_tides[columns_to_keep_sanity]

/opt/anaconda3/envs/tides_compas/lib/python3.11/site-packages/astropy/units/quantity.py:671: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/opt/anaconda3/envs/tides_compas/lib/python3.11/site-packages/astropy/units/quantity.py:671: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)


In [74]:
st_change_idx = np.where((df_sanity_tides_short['StellarType_1'] != df_sanity_tides_short['StellarType_1'].shift()) | (df_sanity_tides_short['StellarType_2'] != df_sanity_tides_short['StellarType_2'].shift()))[0]
st_changes_with_buffer = np.concatenate([st_change_idx, st_change_idx-1])
st_changes_with_buffer.sort()
df_sanity_tides_short.iloc[st_changes_with_buffer][1:]

,Time,M1,M2,R1,R2,SemiMajorAxis_Rsun,ecc,StellarType_1,StellarType_2,Period1,Period2,Period_orb,J1_after,J2_after,Jorb_after,dM1_MT,dM2_MT,ImK22_1,ImK22_2,ImK12_1_dyn,ImK12_2_dyn,ImK12_1_eq,ImK12_2_eq
0,0.00000,26.70760,24.13840,4.800230,4.521210,502.452736,3.941130e-02,1,1,inf,inf,182.957981,0.000000e+00,0.000000e+00,867.6690,0.000,0.00000,5.169080e-11,3.846470e-11,8.140790e-12,6.057810e-12,0,0
1288,7.30215,26.37570,23.93700,11.701300,10.037000,507.779083,3.941130e-02,1,1,2.474882e+11,2.161911e+11,186.856441,7.242330e-11,5.536070e-11,858.7440,0.000,0.00000,5.360000e-17,2.817450e-16,8.441480e-18,4.437200e-17,0,0
1289,7.30215,26.37560,23.93700,11.495300,10.037000,507.781233,3.941130e-02,2,1,1.673253e+11,2.161891e+11,186.857962,7.242280e-11,5.536120e-11,858.7420,0.000,0.00000,5.857600e-17,2.817400e-16,9.225150e-18,4.437120e-17,0,0
2368,7.31306,26.37480,23.93640,35.714000,10.071500,507.794135,3.941130e-02,2,1,1.594590e+12,2.176571e+11,186.867091,7.241840e-11,5.536560e-11,858.7190,0.000,0.00000,2.200730e-19,2.715760e-16,3.465930e-20,4.277060e-17,0,0
2369,7.31306,26.37470,23.93640,35.712200,10.071500,507.796285,3.941130e-02,4,1,1.594435e+12,2.176510e+11,186.868613,7.241750e-11,5.536650e-11,858.7150,0.000,0.00000,2.201250e-19,2.715670e-16,3.466740e-20,4.276920e-17,0,0
4118,7.87954,26.30450,23.90580,189.173000,12.506800,508.815538,3.941130e-02,4,1,4.042304e+13,3.316680e+11,187.620255,7.182660e-11,5.595740e-11,857.0530,0.000,0.00000,2.761620e-22,3.104990e-17,4.349270e-23,4.890050e-18,0,0
4119,7.87960,9.95851,25.01890,1.022830,11.195800,1254.579308,0.000000e+00,7,1,4.046281e+13,1.949899e+00,870.338032,1.277930e-15,7.982270e+00,639.3620,-16.346,1.11311,5.203730e-19,-2.681580e-11,8.195360e-20,-2.689610e-11,0,0
4211,8.00123,9.95271,25.01100,0.933968,11.750800,1255.142693,0.000000e+00,7,1,3.384927e+13,2.151937e+00,871.094549,1.272970e-15,7.965290e+00,639.0540,0.000,0.00000,0.000000e+00,-1.270810e-11,0.000000e+00,-1.275010e-11,0,0
4212,8.00123,9.95260,25.01080,0.933966,11.750800,1255.151294,0.000000e+00,8,1,1.067913e+13,2.151998e+00,871.107775,1.272680e-15,7.964990e+00,639.0480,0.000,0.00000,0.000000e+00,-1.270710e-11,0.000000e+00,-1.274910e-11,0,0
4338,8.02915,9.95090,25.00910,1.067170,11.889200,1255.286764,0.000000e+00,8,1,1.163529e+13,2.203806e+00,871.292981,1.267630e-15,7.961390e+00,638.9620,0.000,0.00000,0.000000e+00,-1.063020e-11,0.000000e+00,-1.066620e-11,0,0


### Notides

In [69]:
RawData = df_detailed_notides
tf = tempfile.TemporaryFile()
Data = h5.File(tf, 'w')
maskRecordType4 = RawData['Record_Type'] == 4     # Filter first for only end-of-timestep events
for key in RawData.keys():
    Data.create_dataset(key, data=RawData[key][maskRecordType4])

### Collect the important events in the detailed evolution
events = det_evo.allEvents(Data).allEvents                 # Calculate the events here, for use in plot sizing parameters
det_evo.printEvolutionaryHistory(events=events)
# det_evo.makeDetailedPlots(Data, events, outdir=None, show=True)
print("Merges Hubble Time?:", df_dco_tides['Merges_Hubble_Time'].values.astype(bool))
det_evo.plotVanDenHeuvel(events, outdir=None, use_latex=True)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/veomekapil/Library/CloudStorage/OneDrive-JohnsHopkins/Research/tides_paper_2/COMPAS/compas_python_utils/detailed_evolution_plotter/van_den_heuvel_figures/2.png'

In [ ]:
df_sanity_notides = pd.read_csv(notides_DataPath+'/sanity_checks.csv', header=0)
df_sanity_notides = df_sanity_notides.rename(columns=lambda x: x.strip())

df_sanity_notides['SemiMajorAxis_Rsun'] = (df_sanity_notides['SemiMajorAxis_after'].values * u.AU).to(u.R_sun)
df_sanity_notides['Period1'] = (2 * np.pi / (df_sanity_notides['Omega1'].values / u.yr)).to(u.day)
df_sanity_notides['Period2'] = (2 * np.pi / (df_sanity_notides['Omega2'].values / u.yr)).to(u.day)
df_sanity_notides['Period_orb'] = (2 * np.pi / (df_sanity_notides['Omega'].values / u.yr)).to(u.day)

df_sanity_notides_short = df_sanity_notides[columns_to_keep_sanity]

/opt/anaconda3/envs/tides_compas/lib/python3.11/site-packages/astropy/units/quantity.py:671: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/opt/anaconda3/envs/tides_compas/lib/python3.11/site-packages/astropy/units/quantity.py:671: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)


In [44]:
st_change_idx = np.where((df_sanity_notides_short['StellarType_1'] != df_sanity_notides_short['StellarType_1'].shift()) | (df_sanity_notides_short['StellarType_2'] != df_sanity_notides_short['StellarType_2'].shift()))[0]
st_changes_with_buffer = np.concatenate([st_change_idx, st_change_idx-1])
st_changes_with_buffer.sort()
df_sanity_notides_short.iloc[st_changes_with_buffer][1:]

NameError: name 'df_sanity_notides_short' is not defined

### Perfect Tides

In [70]:
RawData = df_detailed_perfect
tf = tempfile.TemporaryFile()
Data = h5.File(tf, 'w')
maskRecordType4 = RawData['Record_Type'] == 4     # Filter first for only end-of-timestep events
for key in RawData.keys():
    Data.create_dataset(key, data=RawData[key][maskRecordType4])

### Collect the important events in the detailed evolution
events = det_evo.allEvents(Data).allEvents                 # Calculate the events here, for use in plot sizing parameters
det_evo.printEvolutionaryHistory(events=events)
# det_evo.makeDetailedPlots(Data, events, outdir=None, show=True)
# print("Merges Hubble Time?:", df_dco_perfect['Merges_Hubble_Time'].values.astype(bool))
det_evo.plotVanDenHeuvel(events, outdir=None, use_latex=True)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/veomekapil/Library/CloudStorage/OneDrive-JohnsHopkins/Research/tides_paper_2/COMPAS/compas_python_utils/detailed_evolution_plotter/van_den_heuvel_figures/2.png'

In [75]:
df_sanity_perfect = pd.read_csv(perfect_DataPath+'/sanity_checks.csv', header=0)
df_sanity_perfect = df_sanity_perfect.rename(columns=lambda x: x.strip())

df_sanity_perfect['SemiMajorAxis_Rsun'] = (df_sanity_perfect['SemiMajorAxis_after'].values * u.AU).to(u.R_sun)
df_sanity_perfect['Period1'] = (2 * np.pi / (df_sanity_perfect['Omega1'].values / u.yr)).to(u.day)
df_sanity_perfect['Period2'] = (2 * np.pi / (df_sanity_perfect['Omega2'].values / u.yr)).to(u.day)
df_sanity_perfect['Period_orb'] = (2 * np.pi / (df_sanity_perfect['Omega'].values / u.yr)).to(u.day)

df_sanity_perfect_short = df_sanity_perfect[columns_to_keep_sanity]

In [76]:
st_change_idx = np.where((df_sanity_perfect_short['StellarType_1'] != df_sanity_perfect_short['StellarType_1'].shift()) | (df_sanity_perfect_short['StellarType_2'] != df_sanity_perfect_short['StellarType_2'].shift()))[0]
st_changes_with_buffer = np.concatenate([st_change_idx, st_change_idx-1])
st_changes_with_buffer.sort()
df_sanity_perfect_short.iloc[st_changes_with_buffer][1:]

,Time,M1,M2,R1,R2,SemiMajorAxis_Rsun,ecc,StellarType_1,StellarType_2,Period1,Period2,Period_orb,J1_after,J2_after,Jorb_after,dM1_MT,dM2_MT,ImK22_1,ImK22_2,ImK12_1_dyn,ImK12_2_dyn,ImK12_1_eq,ImK12_2_eq
0,0.00000,26.70760,24.13840,4.800230,4.521210,501.637764,0,1,1,182.512739,182.512739,182.512739,1.673520e-02,1.341800e-02,867.6380,0.000,0.00000,5.169080e-11,3.846470e-11,8.140790e-12,6.057810e-12,0,0
1288,7.30215,26.37570,23.93700,11.701300,10.037000,506.800686,0,1,1,186.317897,186.317897,186.317897,9.620110e-02,6.423700e-02,858.5840,0.000,0.00000,-1.532980e-24,1.118090e-27,-8.574060e-18,-4.470280e-17,0,0
1289,7.30215,26.37560,23.93700,11.495300,10.037000,506.839392,0,2,1,186.339077,186.339077,186.339077,6.503310e-02,6.422960e-02,858.6130,0.000,0.00000,-8.294350e-18,1.281810e-31,-5.579090e-17,-4.471420e-17,0,0
2368,7.31306,26.37480,23.93640,35.714000,10.071500,506.196446,0,2,1,185.985707,185.985707,185.985707,6.208950e-01,6.479320e-02,858.0340,0.000,0.00000,2.091030e-27,5.154410e-31,-3.491630e-20,-4.331210e-17,0,0
2369,7.31306,26.37470,23.93640,35.712200,10.071500,506.198596,0,4,1,185.988721,185.988721,185.988721,6.208210e-01,6.479220e-02,858.0300,0.000,0.00000,-1.748460e-30,2.358360e-30,-3.512000e-20,-4.331040e-17,0,0
4118,7.87954,26.30450,23.90580,189.173000,12.506800,488.602515,0,4,1,176.552354,176.552354,176.552354,1.644530e+01,1.051210e-01,840.5110,0.000,0.00000,3.234210e-30,1.225750e-28,-5.086810e-23,-5.748330e-18,0,0
4119,7.87960,9.95851,25.01890,1.022830,11.195800,1286.032062,0,7,1,903.275278,903.275278,903.275278,5.724570e-05,1.723150e-02,647.3260,-16.346,1.11305,-1.993810e-17,-2.776370e-11,-2.745580e-17,-2.784580e-11,0,0
4211,8.00123,9.95271,25.01090,0.933968,11.750900,1286.533087,0,7,1,903.979767,903.979767,903.979767,4.766610e-05,1.896170e-02,646.9950,0.000,0.00000,0.000000e+00,8.101520e-30,0.000000e+00,-2.034600e-19,0,0
4212,8.00123,9.95260,25.01080,0.933966,11.750900,1286.541688,0,8,1,903.994010,903.994010,903.994010,1.503450e-05,1.896110e-02,646.9890,0.000,0.00000,0.000000e+00,6.094340e-32,0.000000e+00,-2.035060e-19,0,0
4338,8.02915,9.95090,25.00910,1.067170,11.889200,1286.666407,0,8,1,904.164966,904.164966,904.164966,1.631250e-05,1.940520e-02,646.8970,0.000,0.00000,0.000000e+00,4.762580e-29,0.000000e+00,-1.812190e-19,0,0
